<a href="https://colab.research.google.com/github/anawag/pandas-numpy/blob/main/censo_2022.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Import das bases e criação dos dataframes

In [3]:
from google.colab import auth
auth.authenticate_user()

In [6]:
# @title
from google.cloud import bigquery

client = bigquery.Client(project="censo-2022-480100")

alfabetizados = """
SELECT *
FROM `basedosdados.br_ibge_censo_2022.alfabetizacao_grupo_idade_sexo_raca`
"""

esgoto = """
SELECT *
FROM `basedosdados.br_ibge_censo_2022.caracteristica_domicilio_grupo_idade_raca_esgotamento_sanitario`
"""

lixo = """
SELECT *
FROM `basedosdados.br_ibge_censo_2022.caracteristica_domicilio_grupo_idade_raca_destino_lixo`
"""

agua = """
SELECT *
FROM `basedosdados.br_ibge_censo_2022.caracteristica_domicilio_grupo_idade_raca_ligacao_abastecimento_agua`
"""

municipio = """
SELECT *
FROM `basedosdados.br_ibge_censo_2022.municipio`
"""

idade_genero = """
SELECT *
FROM basedosdados.br_ibge_censo_2022.populacao_idade_sexo
"""

alfabetizados_df = client.query(alfabetizados).to_dataframe()
esgoto_df = client.query(esgoto).to_dataframe()
lixo_df = client.query(lixo).to_dataframe()
agua_df = client.query(agua).to_dataframe()
municipio_df = client.query(municipio).to_dataframe()
idade_genero_df = client.query(idade_genero).to_dataframe()


Alfabetizados

In [39]:
rename_alfabetizacao = {
    "cor_raca": "ds_corRaca",
    "sexo": "ds_sexo",
    "grupo_idade": "ds_grupoIdade",
    "alfabetizacao": "ds_alfabetizacao",
    "populacao": "qt_vidasOficial"
}

dtype_alfabetizacao = {
    "id_municipio": "Int64",
    "ds_corRaca": "string",
    "ds_sexo": "string",
    "ds_grupoIdade": "string",
    "ds_alfabetizacao": "string",
    "qt_vidasOficial": "int64"
}

alfabetizados_df = (
    alfabetizados_df
        .rename(columns=rename_alfabetizacao)
        .astype(dtype_alfabetizacao)
)

nulos = alfabetizados_df.columns[alfabetizados_df.isna().any()].tolist() # identificando a coluna com valores nulos

alfabetizados_df["qt_vidasOficial"] = alfabetizados_df["qt_vidasOficial"].fillna(0).astype("int64") # transformando os valores nulos em 0

for col in alfabetizados_df.columns: # descobrindo dados duplicados
  if alfabetizados_df[col].duplicated().any():
     print(f"A coluna {col} possui valores duplicados")
  else:
     print(f"A coluna {col} não possui valores duplicados")

id_duplicado_alfa = alfabetizados_df["id_municipio"].duplicated() # descobrindo quais ids estão duplicados

alfabetizados_df = alfabetizados_df.drop_duplicates(subset="id_municipio", keep="first") # mantendo só a primeira vez que o id surge


for col in alfabetizados_df.columns:
    if alfabetizados_df[col].dtype == "string":
        alfabetizados_df[col] = alfabetizados_df[col].str.upper().str.normalize('NFKD').str.encode(
            'ascii', errors='ignore').str.decode('utf-8') # tranformando em maiúsculas e removendo acentos

print(f"\nTabela Alfabetizados")

print(f"\n {alfabetizados_df}")

A coluna id_municipio não possui valores duplicados
A coluna ds_corRaca possui valores duplicados
A coluna ds_sexo possui valores duplicados
A coluna ds_grupoIdade possui valores duplicados
A coluna ds_alfabetizacao possui valores duplicados
A coluna qt_vidasOficial possui valores duplicados

Tabela Alfabetizados

        id_municipio ds_corRaca   ds_sexo ds_grupoIdade   ds_alfabetizacao  \
0           1100023   INDIGENA    HOMENS  15 A 19 ANOS  NAO ALFABETIZADAS   
1           1100262    AMARELA  MULHERES  15 A 19 ANOS  NAO ALFABETIZADAS   
2           1101005    AMARELA  MULHERES  15 A 19 ANOS  NAO ALFABETIZADAS   
3           1101435    AMARELA  MULHERES  15 A 19 ANOS  NAO ALFABETIZADAS   
5           1100346   INDIGENA  MULHERES  15 A 19 ANOS  NAO ALFABETIZADAS   
...             ...        ...       ...           ...                ...   
48767       3106200    AMARELA    HOMENS  15 A 19 ANOS  NAO ALFABETIZADAS   
49902       5208707   INDIGENA  MULHERES  15 A 19 ANOS  NAO ALFABET

In [40]:
rename_esgoto = {
"ano": "nr_ano",
"tipo_esgotamento": "tp_esgotamento",
"grupo_idade": "ds_grupoIdade",
"cor_raca": "ds_corRaca",
"populacao": "qt_vidasOficial"
}


dtype_esgoto = {
  "nr_ano" : "Int64",
  "id_municipio" : "Int64",
  "tp_esgotamento" : "string",
  "ds_grupoIdade" : "string",
  "ds_corRaca" : "string",
}

esgoto_df = (
    esgoto_df
        .rename(columns=rename_esgoto)
        .astype(dtype_esgoto)
)


nulos = esgoto_df.columns[esgoto_df.isna().any().tolist()]

esgoto_df["qt_vidasOficial"] = esgoto_df["qt_vidasOficial"].fillna(0).astype("int64")

for col in esgoto_df:
  if esgoto_df[col].duplicated().any():
    print(f"A coluna {col} está sendo duplicada")
  else:
    print(f"A coluna {col} não está sendo duplicada")

id_duplicado_esgoto = esgoto_df["id_municipio"].duplicated()
print(f"\n {id_duplicado_esgoto}")

esgoto_df = esgoto_df.drop_duplicates(subset="id_municipio", keep="first")

for col in esgoto_df:
  if esgoto_df[col].dtype == "string":
    esgoto_df[col] = esgoto_df[col].str.upper().str.normalize('NFKD').str.encode(
        'ascii', errors='ignore').str.decode('utf-8')

print(f"\nTabela Esgotamento Sanitário")

print(f"\n {esgoto_df}")

A coluna nr_ano está sendo duplicada
A coluna id_municipio não está sendo duplicada
A coluna tp_esgotamento está sendo duplicada
A coluna ds_grupoIdade está sendo duplicada
A coluna ds_corRaca está sendo duplicada
A coluna qt_vidasOficial está sendo duplicada

 0        False
1        False
2        False
3        False
4        False
         ...  
32945    False
33592    False
34609    False
37244    False
63537    False
Name: id_municipio, Length: 5570, dtype: bool

Tabela Esgotamento Sanitário

        nr_ano  id_municipio                                   tp_esgotamento  \
0        2022       1100056                       FOSSA RUDIMENTAR OU BURACO   
1        2022       1100114      FOSSA SEPTICA OU FOSSA FILTRO LIGADA A REDE   
2        2022       1100205  REDE GERAL, REDE PLUVIAL OU FOSSA LIGADA A REDE   
3        2022       1100304                       FOSSA RUDIMENTAR OU BURACO   
4        2022       1100379      FOSSA SEPTICA OU FOSSA FILTRO LIGADA A REDE   
...       ...  

In [ ]:
rename_lixo = {
"ano": "nr_ano",
"destino_lixo": "tp_destinoLixo",
"grupo_idade": "ds_grupoIdade",
"cor_raca": "ds_corRaca",
"populacao": "qt_vidasOficial"
}

dtype_lixo = {
    "nr_ano": "Int64",
    "id_municipio" : "string",
    "tp_destinoLixo": "string",
    "ds_grupoIdade" : "string",
    "ds_corRaca" : "string"
}

lixo_df = (
    lixo_df
      .rename(columns=rename_lixo)
        .astype(dtype_lixo)
  )


nulo = lixo_df.columns[lixo_df.isna().any()].tolist()

lixo_df["qt_vidasOficial"] = lixo_df["qt_vidasOficial"].fillna(0).astype("int64")

for col in lixo_df:
  if lixo_df[col].duplicated().any():
    print(f"A coluna {col} está sendo duplicada")
  else:
    print(f"A coluna {col} não está sendo duplicada")

id_duplicado_lixo = lixo_df["id_municipio"].duplicated()
print(f"\n {id_duplicado_lixo}")

lixo_df = lixo_df.drop_duplicates(subset="id_municipio", keep="first")

for col in lixo_df:
  if lixo_df[col].dtype == "string":
    lixo_df[col] = lixo_df[col].str.upper().str.normalize('NFKD').str.encode(
        'ascii', errors='ignore').str.decode('utf-8')


In [44]:
print(lixo_df)

       nr_ano id_municipio                                tp_destinoLixo  \
0        2022      1715150                                      COLETADO   
1        2022      2414308                                      COLETADO   
2        2022      3111309                                      COLETADO   
3        2022      5205703                                      COLETADO   
4        2022      2708006                                      COLETADO   
...       ...          ...                                           ...   
43231    2022      4305959                                 OUTRO DESTINO   
45681    2022      4121257                      ENTERRADO NA PROPRIEDADE   
48535    2022      3204658   DEPOSITADO EM CACAMBA DE SERVICO DE LIMPEZA   
53731    2022      5102793  COLETADO NO DOMICILIO POR SERVICO DE LIMPEZA   
53800    2022      5107198  COLETADO NO DOMICILIO POR SERVICO DE LIMPEZA   

          ds_grupoIdade ds_corRaca  qt_vidasOficial  
0            0 A 4 ANOS    AMAREL